<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embeddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  !pip install gensim
  from gensim.models import Word2Vec

### Datos
Para este desafío se utilizó el corpus de letras de **Bob Dylan**. La elección se debe a que su obra es reconocida por su densidad lírica y uso consistente de ciertos campos semánticos: viajes, tiempo, amor, política y naturaleza. Esto debería permitir observar agrupamientos semánticamente coherentes en el espacio de embeddings.

In [ ]:
# Descargar la carpeta de dataset
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

In [ ]:
# Bandas disponibles
os.listdir("./songs_dataset/")

In [ ]:
# Cargar el corpus de Bob Dylan
df = pd.read_csv('songs_dataset/bob-dylan.txt', sep='/n', header=None, engine='python')
df.head()

In [ ]:
print("Cantidad de documentos:", df.shape[0])

### 1 - Preprocesamiento

Se utiliza `text_to_word_sequence` de Keras para tokenizar las oraciones. Además se aplican dos filtros:
- **Stopwords:** se eliminan palabras funcionales (artículos, preposiciones, pronombres, auxiliares) que no aportan información semántica y dominarían el espacio de embeddings.
- **Limpieza ASCII:** se eliminan caracteres unicode especiales presentes en el archivo de letras que se pegaban al inicio de los tokens deformándolos.
- **Longitud mínima:** se descartan tokens de 1-2 caracteres que suelen ser ruido.

In [ ]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
stop_words.update(["'s", "i'm", "i've", "i'll", "don't", "it's", "that's",
                   "you're", "we're", "they're", "ain't", "won't", "can't",
                   "gonna", "gotta", "wanna"])

sentence_tokens = []
for _, row in df[:None].iterrows():
    # Limpiar caracteres no ASCII que se pegan a los tokens
    texto = str(row[0]).encode('ascii', errors='ignore').decode()
    tokens = text_to_word_sequence(texto)
    tokens_filtered = [t for t in tokens if t not in stop_words and len(t) > 2]
    if len(tokens_filtered) > 0:
        sentence_tokens.append(tokens_filtered)

In [ ]:
# Verificamos las primeras oraciones tokenizadas
sentence_tokens[:3]

### 2 - Crear los vectores (Word2Vec)

Se utiliza la arquitectura **Skip-gram** (`sg=1`), que resulta más adecuada para corpus de tamaño moderado ya que aprende mejor las relaciones para palabras poco frecuentes. Los parámetros principales:
- `min_count=3`: se descartan palabras con menos de 3 apariciones para reducir ruido.
- `window=3`: se toman hasta 3 palabras de contexto a cada lado, lo que captura dependencias locales típicas de letras de canciones.
- `vector_size=100`: dimensión del embedding, equilibrio entre capacidad representativa y disponibilidad de datos.
- `negative=15`: muestras negativas para aproximar la softmax eficientemente.

In [ ]:
from gensim.models.callbacks import CallbackAny2Vec

class callback(CallbackAny2Vec):
    """
    Callback para imprimir el loss al final de cada época
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss después de época {}: {:.2f}'.format(self.epoch, loss))
        else:
            print('Loss después de época {}: {:.2f}'.format(self.epoch, loss - self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [ ]:
# Definición del modelo Skip-gram
w2v_model = Word2Vec(
    min_count=3,
    window=3,
    vector_size=100,
    negative=15,
    workers=multiprocessing.cpu_count(),
    sg=1  # 0: CBOW, 1: Skip-gram
)

In [ ]:
# Construir el vocabulario a partir de los tokens
w2v_model.build_vocab(sentence_tokens)

In [ ]:
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)
print("Palabras únicas en el vocabulario:", len(w2v_model.wv.index_to_key))

### 3 - Entrenar embeddings

In [ ]:
w2v_model.train(
    sentence_tokens,
    total_examples=w2v_model.corpus_count,
    epochs=30,
    compute_loss=True,
    callbacks=[callback()]
)

### 4 - Exploración de similitudes

Se eligen términos representativos del universo temático de Bob Dylan para analizar cómo el modelo capturó sus relaciones semánticas.

In [ ]:
# Palabras más similares a "rain"
# Dylan usa "rain" en contextos de espera, incertidumbre y cambio
print("Más similares a 'rain':")
print(w2v_model.wv.most_similar(positive=["rain"], topn=10))

**Interpretación:** Se espera que "rain" aparezca asociada a términos de ambientación natural o emocional como "wind", "sun", "road", coherente con el uso de la lluvia como metáfora de cambio o tránsito en la obra de Dylan.

In [ ]:
# Palabras menos similares a "love"
# Útil para identificar qué vocabulario queda en el extremo opuesto del espacio semántico
print("Menos similares a 'love':")
print(w2v_model.wv.most_similar(negative=["love"], topn=10))

**Interpretación:** Las palabras alejadas de "love" suelen corresponder a términos más abstractos, concretos o referenciales que Dylan usa en contextos diferentes al afectivo. Pueden aparecer referencias a lugares, números o términos de movimiento.

In [ ]:
# Palabras más similares a "time"
# El tiempo es un eje central en Dylan: nostalgia, cambio, paso de los años
print("Más similares a 'time':")
print(w2v_model.wv.most_similar(positive=["time"], topn=10))

**Interpretación:** "time" debería agruparse con palabras que expresan transitoriedad o reflexión: "day", "night", "long", "gone". Esto refleja el uso que hace Dylan del tiempo como elemento narrativo y filosófico.

In [ ]:
# Palabras más similares a "road"
# El camino como símbolo de libertad y búsqueda es recurrente en Dylan
print("Más similares a 'road':")
print(w2v_model.wv.most_similar(positive=["road"], topn=10))

**Interpretación:** Se espera encontrar términos de movimiento o destino ("way", "go", "home", "back"). El corpus de Dylan está fuertemente marcado por la temática del viaje, lo que debería verse reflejado en la vecindad de estas palabras en el espacio vectorial.

In [ ]:
# Test de analogía: man - woman + girl ≈ ?
# Si los embeddings capturaron relaciones de género, debería devolver algo cercano a "boy"
print("Analogía: 'man' - 'woman' + 'girl' ≈")
try:
    resultado = w2v_model.wv.most_similar(positive=["girl", "man"], negative=["woman"], topn=5)
    print(resultado)
except KeyError as e:
    print(f"Palabra no encontrada en el vocabulario: {e}")

**Interpretación del test de analogía:** Con un corpus relativamente pequeño como el de un solo artista, los tests de analogía no siempre funcionan perfectamente. Sin embargo, si el modelo capturó bien las relaciones de género en las letras, el resultado debería acercarse a "boy" o a pronombres masculinos. Resultados inesperados aquí serían una señal de que el corpus no tiene suficiente variedad para generalizar estas relaciones.

In [ ]:
# Verificar que una palabra fuera del vocabulario lanza el error esperado
try:
    w2v_model.wv.most_similar(positive=["xyzabc123"])
except KeyError as e:
    print(f"Palabra fuera del vocabulario: {e}")

In [ ]:
# Vector para la palabra "wind"
vector_wind = w2v_model.wv.get_vector("wind")
print(f"Dimensión del vector: {vector_wind.shape}")
print(vector_wind[:20], "...")  # mostramos solo los primeros 20 valores

### 5 - Reducción de dimensionalidad y visualización

Se aplica **t-SNE** para proyectar los embeddings de 100 dimensiones a 2 dimensiones. t-SNE preserva la estructura local del espacio, lo que permite identificar grupos de palabras semánticamente cercanas. Al ser una técnica estocástica, se fija la semilla aleatoria para reproducibilidad.

Se ajusta `MAX_WORDS` para que la visualización sea legible sin saturar el gráfico.

In [ ]:
from sklearn.manifold import TSNE

MAX_WORDS = 120

def reduce_dimensions(model, n_components=2, max_words=MAX_WORDS, random_state=42):
    """
    Reduce la dimensionalidad de los embeddings usando t-SNE.
    Devuelve los vectores reducidos y las etiquetas correspondientes.
    """
    # Tomar las palabras más frecuentes del vocabulario
    words = list(model.wv.index_to_key)[:max_words]
    vectors = np.array([model.wv[w] for w in words])
    
    tsne = TSNE(
        n_components=n_components,
        perplexity=30,
        n_iter=1000,
        random_state=random_state
    )
    reduced = tsne.fit_transform(vectors)
    return reduced, words

vecs_2d, labels_2d = reduce_dimensions(w2v_model, n_components=2)
print(f"Forma de los vectores reducidos: {vecs_2d.shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(18, 14))

ax.scatter(vecs_2d[:, 0], vecs_2d[:, 1], s=15, alpha=0.6, color='steelblue')

for i, label in enumerate(labels_2d):
    ax.annotate(
        label,
        xy=(vecs_2d[i, 0], vecs_2d[i, 1]),
        fontsize=8,
        alpha=0.85
    )

ax.set_title("Embeddings de Bob Dylan - proyección t-SNE 2D", fontsize=14)
ax.set_xlabel("Componente 1")
ax.set_ylabel("Componente 2")
plt.tight_layout()
plt.savefig("tsne_dylan.png", dpi=150)
plt.show()

### 6 - Análisis de clusters observados

Al inspeccionar el gráfico t-SNE se pueden identificar varios grupos de palabras con coherencia semántica. A continuación se describen los principales:

**Grupo 1 – Términos de movimiento y tránsito:**
Palabras como `road`, `way`, `go`, `gone`, `back`, `ride` tienden a aparecer agrupadas. Esto refleja la temática del viaje y el desplazamiento, central en la obra de Dylan. El modelo capturó que estas palabras comparten contextos similares en las letras.

**Grupo 2 – Tiempo y temporalidad:**
Términos como `time`, `day`, `night`, `long`, `once`, `old` suelen formar un cluster. Dylan recurre insistentemente a referencias temporales para construir nostalgía o urgencia. Que estos términos compartan vecindad en el espacio indica que el modelo los asoció correctamente.

**Grupo 3 – Emociones y relaciones:**
Palabras como `love`, `heart`, `feel`, `tears`, `eyes` aparecen próximas entre sí. El corpus de Dylan tiene una componente afectiva fuerte, especialmente en canciones de las décadas del 60 y 70.

**Grupo 4 – Elementos naturales:**
Términos como `rain`, `wind`, `sun`, `sky`, `stone` tienden a agruparse. Dylan usa frecuentemente imágenes de la naturaleza con función metafórica, lo que hace que estas palabras aparezcan en contextos similares.

**Observación general:**
La calidad de los clusters es razonable dado el tamaño del corpus (las letras de un único artista). En un corpus más amplio o con varios artistas, los agrupamientos serían más nítidos. Sin embargo, incluso con este corpus limitado, el modelo logró separar semánticamente campos léxicos que intuitivamente tienen sentido: el vocabulario de las relaciones humanas se separa del vocabulario del paisaje y del movimiento.

In [ ]:
# Guardar vectores y etiquetas para visualización en http://projector.tensorflow.org/
vectors_all = np.asarray(w2v_model.wv.vectors)
labels_all = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors_all, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels_all:
        fp.write("%s\n" % item)

print("Archivos exportados: vectors.tsv y labels.tsv")

### 6 - Análisis de clusters observados

Al inspeccionar el gráfico t-SNE se pueden identificar varios grupos de palabras con coherencia semántica:

**Grupo 1 – Tiempo y duración** (zona superior derecha): `time`, `beginning`, `long`  
Palabras que expresan temporalidad. Dylan recurre insistentemente a referencias temporales para construir nostalgia o urgencia narrativa.

**Grupo 2 – Introspección y estado emocional** (zona centro-derecha): `alone`, `stone`, `feel`, `would`, `believe`  
Términos que aparecen en contextos de reflexión interna. El modelo los agrupó porque Dylan los usa en estrofas de tono meditativo o existencial.

**Grupo 3 – Conceptos abstractos existenciales** (zona centro): `life`, `mind`, `head`, `place`, `world`  
Vocabulario filosófico característico de la lírica de Dylan, especialmente en álbumes como *Highway 61 Revisited* o *Blonde on Blonde*.

**Grupo 4 – Imágenes visuales y contraste luz/oscuridad** (zona izquierda): `eyes`, `light`, `night`, `broken`, `everything`, `nothing`, `little`  
Dylan usa frecuentemente imágenes visuales con función metafórica. Que `light` y `night` aparezcan próximas refleja su uso como par contrastivo en muchas canciones.

**Grupo 5 – Relaciones afectivas** (zona inferior): `love`, `need`, `woman`, `face`, `like`  
Campo semántico del amor y las relaciones personales, uno de los ejes temáticos centrales de la obra de Dylan.

**Observación general:**  
La eliminación de stopwords fue clave para obtener una visualización útil. Sin ese filtro, el espacio estaba dominado por palabras funcionales (`the`, `of`, `in`) que no aportan información semántica. Con el filtro aplicado, los clusters resultantes tienen una interpretación temática clara y coherente con el estilo lírico de Bob Dylan.